# Conference Fig. 4 — ChargE3Net density on simulated PCM arrays

**Claim.** Averaging the predictions of three independently programmed arrays keeps the ChargE3Net density at ≈1 % NMAE, and non-self-consistent runs from these densities reproduce the DFT DOS, with band gaps within 42 meV of SCF over the Ge–Sb–Te series (and for a DiffCSP-generated Ge₂Sb₂Te₅).

- **a** NMAE vs DFT, analog (prediction averaged over 3 arrays; bar = single-array range) vs digital; solid: analog = digital ⊕ a structure-independent hardware term, √(x² + σ_hw²) with σ_hw = 0.74 % (`results/D1/probes300k_dual_am_c012/hw_floor.json`, `scripts/d1_hw_floor.py`); text: NMAE of the analog vs the digital prediction, median (range): 100 random ChargE3Net test structures (3×10⁵ grid points each) and the Ge–Sb–Te series (full VASP grid) — `results/D1/probes300k_dual_am_c012_shard*of2/per_structure.csv`, `results/D3/final/nmae.csv`.
- **b** Ge₂Sb₂Te₅ (mp-1224375): DFT density on an (a, c) plane and the digital / analog errors — `results/D3/final/slices_mp-1224375.npz`.
- **c** NMAE vs DFT of the Ge–Sb–Te series (full VASP grid): digital vs analog on a single array (mean ± s.d. over the 3 arrays) — `results/D3/final/nmae.csv`.
- **d** Total DOS of Te, Sb₂Te₃, Ge₂Sb₂Te₅ and Ge₃Sb₂Te₆ within ±2 eV of the VBM (2 × 2, drawn as Fig. S top row): DFT (SCF, shaded) vs NSCF runs from the digital and analog densities; dotted = VBM — `results/D3/nscf/dos/dos.csv`.

"analog" everywhere = the nominal ChargE3Net result: prediction averaged over 3 independently programmed arrays (chip seeds 0–2). The projected energy moved to Fig. 3a.
Caveats: GST compounds are in the ChargE3Net / MP training data.


In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
from style import *
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
W = W_CONF          # 会议版画布宽度 178 mm
from pymatgen.core import Composition
D1 = RESULTS / 'D1'
FINAL = RESULTS / 'D3' / 'final'
D1_RUN = 'probes300k_dual_am_c012'     # 名义：三份独立编程阵列（芯片种子 0–2）的平均预测
NSCF_D3 = FINAL / 'nscf.csv'
NSCF_S1 = (RESULTS / 'D3/nscf/d3/gpu_dual_am_mean3_lp.csv', RESULTS / 'D3/nscf/d3/gpu_dual_am_lp.csv')
S1_ID = 's1-Ge2Sb2Te5-c2'              # DiffCSP 生成的 Ge2Sb2Te5（S1 链路）
DOSDIR = RESULTS / 'D3' / 'nscf' / 'dos'
NS = RESULTS / 'P1_neurosim' / 'nominal_22nm'
CMP = json.loads((NS / 'compare.json').read_text())
NSM = json.loads((NS / 'metrics.json').read_text())
CHIPS = (0, 1, 2)
D3_IDS = ['mp-32', 'mp-104', 'mp-19', 'mp-938', 'mp-1201', 'mp-29641', 'mp-1224350', 'mp-1224375', 'mp-1212511']
ORDER = ['Ge', 'Sb', 'Te']

def formula_tex(f):
    c = Composition(f).reduced_composition.get_el_amt_dict()
    els = sorted(c, key=lambda e: ORDER.index(e) if e in ORDER else 99)
    return ''.join(e + (f'$_{{{int(c[e])}}}$' if c[e] != 1 else '') for e in els)

def nscf_table():
    # D3 final (9 structures) + S1 generated Ge2Sb2Te5; columns material_id, init, dgap_meV
    t = pd.read_csv(NSCF_D3)[['material_id', 'init', 'dgap_meV']]
    s = pd.concat([pd.read_csv(p) for p in NSCF_S1])
    s = s[(s.material_id == S1_ID) & s.dgap_meV.notna()][['material_id', 'init', 'dgap_meV']].drop_duplicates(['init'])
    return pd.concat([t, s])


In [ ]:
# 字体设置 / fonts —— 全文唯一来源（style.PAPER_FONTS）
FS = use_paper_fonts()
FS_TITLE, FS_LABEL, FS_TICK = FS['title'], FS['label'], FS['tick']
FS_LEGEND, FS_ANNOT, FS_PANEL = FS['legend'], FS['annot'], FS['panel']
FS_CBLABEL, FS_CBTICK = FS['cbar_label'], FS['cbar_tick']


In [ ]:
# layout: one row of 4 panels, canvas W_CONF = 7.008 in (a, b as before)
# Derivation (inches): left 0.38; widths a 0.85 / b 1.15 (3 strips + 2 colour bars) / c 1.45 (NMAE bars) / d 1.70
#   (DOS 2 x 2); gaps 0.42 / 0.52 (b's colour-bar labels + c's y label and ticks) / 0.30 (d's "DOS" label)
#   -> right edge 0.38 + 5.15 + 1.24 = 6.77 in; axes h 1.05, bottom 0.50 (rotated formulas in c), top 0.16 -> H = 1.71 in
OUT = FIG_OUT / 'conf_fig4'
PANEL_OUT = OUT / 'panels'
PREVIEW_SCALE = 0.5
mpl.rcParams['savefig.transparent'] = True   # 导出背景透明（整图与单面板）
H = 1.71

def make_layout():
    fig = plt.figure(figsize=(W, H))
    return fig, row_slots(fig, 0.38, [0.85, 1.15, 1.45, 1.70], [0.42, 0.52, 0.30], bottom=0.50, top=0.16)

def row_slots(fig, left, ax_w, gaps, bottom, top):
    # One row of panels from widths in inches: axes widths `ax_w`, label gaps `gaps` between them.
    ratios = [v for pair in zip(ax_w, list(gaps) + [0]) for v in pair][:-1]
    gs = fig.add_gridspec(1, len(ratios), left=left / W, right=(left + sum(ratios)) / W,
                          bottom=bottom / H, top=1 - top / H, width_ratios=ratios, wspace=0)
    return [gs[2 * k] for k in range(len(ax_w))]

def panel_run(i, letter, draw):
    fig, slots = make_layout()
    draw(fig, slots['abcd'.index(letter)])
    group = {'axes': list(fig.axes), 'texts': list(fig.texts)}
    boxes = save_matched_panels(fig, {letter: group}, PANEL_OUT); plt.close(fig)
    display(NotebookImage(filename=str(PANEL_OUT / f'{letter}.png'), width=int(boxes[letter].width * 600 * PREVIEW_SCALE)))


In [ ]:
# ---------------- 本面板配色（独立可改，不影响其它面板） ----------------
Ca_PT = '#3C5488'      # D1 测试结构（三阵列平均预测）
Ca_RANGE = '#9DB0D3'   # 单阵列范围
Ca_GST = '#E64B35'     # Ge–Sb–Te 系列（全网格）
Ca_GST_RANGE = '#F2A595'
Ca_DIAG = '#7E7E7E'    # y = x
Ca_MODEL = '#0B1B3F'   # y = sqrt(x^2 + sigma_hw^2)
Ca_NOTE = '#3C5488'    # 硬件误差统计（analog vs digital）
# ----------------------------------------------------------------------
def draw_a(fig, spec, model=True, linear=False):
    d = pd.concat([pd.read_csv(p) for p in sorted(D1.glob(f'{D1_RUN}_shard*of2/per_structure.csv'))])
    g = pd.read_csv(FINAL / 'nmae.csv')
    ax = fig.add_subplot(spec)
    sets = ((d.nmae_digital, d.nmae_analog_chipmean, d[[f'nmae_analog_chip{k}' for k in CHIPS]], Ca_PT, Ca_RANGE, 4,
             f'test set (n = {len(d)})'),
            (g.nmae_digital, g.nmae_mean3, g[[f'nmae_chip{k}' for k in CHIPS]], Ca_GST, Ca_GST_RANGE, 6, 'Ge–Sb–Te'))
    for dg, mean, chips, c, cr, s, lab in sets:
        dg, mean, chips = dg.to_numpy() * 100, mean.to_numpy() * 100, chips.to_numpy() * 100
        lo, hi = np.minimum(mean, chips.min(1)), np.maximum(mean, chips.max(1))
        ax.vlines(dg, lo, hi, color=cr, lw=0.5, zorder=1)
        ax.scatter(dg, mean, s=s, color=c, lw=0, zorder=2, label=lab)
    lim = (0, 4) if linear else (0.1, 10)
    ax.plot(lim, lim, color=Ca_DIAG, lw=0.5, ls='--', zorder=0)
    if linear:
        ax.set_xlim(*lim); ax.set_ylim(*lim); ax.set_xticks([0, 1, 2, 3, 4]); ax.set_yticks([0, 1, 2, 3, 4])
        ax.set_aspect('equal', adjustable='box')
    else:
        ax.set_xscale('log'); ax.set_yscale('log'); ax.set_xlim(*lim); ax.set_ylim(*lim)
    ax.set_xlabel('Digital NMAE (%)'); ax.set_ylabel('Analog NMAE (%)')
    ax.legend(loc='upper left', fontsize=FS_LEGEND - 1, handletextpad=0.1)
    if model:   # quadrature model: digital error ⊕ structure-independent hardware term (scripts/d1_hw_floor.py)
        hw = json.loads((D1 / D1_RUN / 'hw_floor.json').read_text()); sg = hw['sigma_hw_pct']
        xx = np.linspace(0, 4, 200) if linear else np.logspace(-1, 1, 200)
        ax.plot(xx, np.sqrt(xx ** 2 + sg ** 2), color=Ca_MODEL, lw=0.8, zorder=3,
                label='$\\sqrt{x^2+\\sigma_{hw}^2}$')   # σ_hw 数值只写在图注里
        ax.legend(loc='upper left', fontsize=FS_LEGEND - 1, handletextpad=0.3, handlelength=1.2)   # 曲线并入图例
        v = hw['analog_vs_digital_pct']
        ax.text(0.97, 0.04, f"analog vs digital\n{v['median']:.2f}% ({v['min']:.2f}–{v['max']:.2f}%)",
                transform=ax.transAxes, ha='right', va='bottom', fontsize=FS_ANNOT - 1, color=Ca_NOTE)
panel_run(0, 'a', draw_a)


In [ ]:
# ---------------- 本面板配色（独立可改，不影响其它面板） ----------------
Cb_RHO = mpl.colors.LinearSegmentedColormap.from_list('rho', ['#FFFFFF', '#9DB0D3', '#3C5488', '#0B1B3F'])
Cb_ERR = mpl.colors.LinearSegmentedColormap.from_list('err', ['#3C5488', '#FFFFFF', '#E64B35'])
Cb_FRAME = '#000000'
# ----------------------------------------------------------------------
def draw_b(fig, spec, mid='mp-1224375', rmin=1e-3, rmax=20, emax=0.3, elin=2e-3):
    s = np.load(FINAL / f'slices_{mid}.npz')
    cell = s['cell']; la, lc = np.linalg.norm(cell[0]), np.linalg.norm(cell[2])
    n_rho = mpl.colors.LogNorm(rmin, rmax)
    n_err = mpl.colors.SymLogNorm(linthresh=elin, linscale=0.6, vmin=-emax, vmax=emax, base=10)
    panels = [('DFT', np.clip(s['dft'], rmin, None), Cb_RHO, n_rho),
              ('Δ dig.', s['digital'] - s['dft'], Cb_ERR, n_err),
              ('Δ ana.', s['analog_mean3'] - s['dft'], Cb_ERR, n_err)]
    # 3 strips + colour-bar column (bars 0.06 in wide, ticks / labels to their right)
    sub = spec.subgridspec(1, 5, width_ratios=[1, 1, 1, 0.22, 0.70], wspace=0.15)
    ims = []
    for k, (title, img, cmap, norm) in enumerate(panels):
        ax = fig.add_subplot(sub[k])
        ims.append(ax.imshow(img.T, origin='lower', extent=(0, la, 0, lc), cmap=cmap, norm=norm,
                             aspect='equal', interpolation='bilinear'))
        ax.set_title(title, fontsize=FS_ANNOT - 1, pad=1.5)
        ax.set_xticks([]); ax.set_yticks([] if k else [0, 5, 10, 15])
        if k == 0:
            ax.set_ylabel('z (Å)')
        for sp in ax.spines.values():
            sp.set_visible(True); sp.set_linewidth(0.4); sp.set_color(Cb_FRAME)
    cb = sub[3].subgridspec(2, 1, hspace=0.45)
    for i, (im, lab) in enumerate(((ims[0], 'ρ (e Å$^{-3}$)'), (ims[2], 'Δρ (e Å$^{-3}$)'))):
        cax = fig.add_subplot(cb[i])
        c = fig.colorbar(im, cax=cax)
        if i == 0:
            c.set_ticks([1e-3, 1e-1, 1e1], labels=['10$^{-3}$', '10$^{-1}$', '10$^{1}$'])
        else:
            c.set_ticks([-0.1, 0, 0.1], labels=['−0.1', '0', '0.1'])
        c.ax.minorticks_off()
        c.ax.tick_params(labelsize=FS_CBTICK, width=0.4, length=1.5, pad=1)
        c.set_label(lab, fontsize=FS_CBLABEL, labelpad=1)
        c.outline.set_linewidth(0.4)
panel_run(1, 'b', draw_b)


In [ ]:
# ---------------- 本面板配色（独立可改，不影响其它面板） ----------------
Cc_DIG = '#7E7E7E'      # 数字
Cc_ANA = '#3C5488'      # 模拟，单阵列（3 块芯片均值 ± 标准差）
Cc_ERR = '#000000'
Cc_HATCH = '#000000'    # 模拟柱的斜线阴影
# ----------------------------------------------------------------------
C_W = 0.38              # bar width
mpl.rcParams['hatch.linewidth'] = 0.4   # 斜线阴影线宽（本图仅 c 用到）

def draw_c(fig, spec):
    t = pd.read_csv(FINAL / 'nmae.csv').set_index('material_id').reindex(D3_IDS)
    ax = fig.add_subplot(spec)
    x = np.arange(len(t))
    ax.bar(x - C_W / 2, t.nmae_digital * 100, C_W, color=Cc_DIG, label='digital')
    ax.bar(x + C_W / 2, t.nmae_chip_mean * 100, C_W, yerr=t.nmae_chip_std * 100, color=Cc_ANA,
           label='analog, single array', error_kw=dict(ecolor=Cc_ERR, lw=0.5, capsize=1.0),
           hatch='//////', edgecolor=Cc_HATCH, lw=0)
    ax.set_xticks(x, [formula_tex(f) for f in t.formula], rotation=60, ha='right', rotation_mode='anchor')
    ax.set_xlim(-0.6, len(t) - 0.4); ax.set_ylim(0, 2.1); ax.set_yticks([0, 0.5, 1.0, 1.5, 2.0])
    ax.set_ylabel('NMAE vs DFT (%)')
    ax.legend(loc='upper left', fontsize=FS_LEGEND - 1, ncol=2, handlelength=0.8, handletextpad=0.3,
              columnspacing=0.6, borderaxespad=0.1)
    return ax
panel_run(2, 'c', draw_c)


In [ ]:
# ---------------- 本面板配色（独立可改，不影响其它面板） ----------------
Cd_DOS = {'scf': dict(label='DFT', color='#E64B35', lw=0.7, ls='-'),        # 与图 S（DOS）第一行一致
          'digital': dict(label='digital', color='#7E7E7E', lw=0.8, ls=(0, (3, 1.5))),
          'analog_mean3': dict(label='analog', color='#3C5488', lw=0.8, ls='-')}
Cd_SCF_FILL = '#E6E6E6'  # DFT DOS 底色
Cd_VBM = '#000000'       # VBM 点线
# ----------------------------------------------------------------------
D_DOS = (('mp-19', 'Te'), ('mp-1201', 'Sb$_2$Te$_3$'), ('mp-1224375', 'Ge$_2$Sb$_2$Te$_5$'),
         ('mp-1212511', 'Ge$_3$Sb$_2$Te$_6$'))   # 2 x 2, row-wise
D_E = (-2.0, 2.0)        # eV vs VBM
D_HEAD = 1.35            # y headroom for the formula / legend

def draw_d(fig, spec):
    dos = pd.read_csv(DOSDIR / 'dos.csv')
    g = spec.subgridspec(2, 2, wspace=0.10, hspace=0.12)
    axs = [fig.add_subplot(g[i // 2, i % 2]) for i in range(4)]
    for ax, (mid, name) in zip(axs, D_DOS):
        top = 0
        for init, st in Cd_DOS.items():
            d = dos[(dos.material_id == mid) & (dos.init == init)]
            m = (d.E_eV >= D_E[0]) & (d.E_eV <= D_E[1])
            e, y = d.E_eV[m].to_numpy(), d.total[m].to_numpy()
            if init == 'scf':
                ax.fill_between(e, 0, y, color=Cd_SCF_FILL, lw=0, zorder=1)
            ax.plot(e, y, color=st['color'], lw=st['lw'], ls=st['ls'], label=st['label'],
                    zorder=5 if init == 'digital' else 3)   # digital 虚线置于最上层
            top = max(top, y.max())
        ax.set_xlim(*D_E); ax.set_ylim(0, top * D_HEAD)
        ax.axvline(0, ymax=1 / D_HEAD, color=Cd_VBM, lw=0.4, ls=':', zorder=2)
        ax.text(0.03, 0.97, name, transform=ax.transAxes, ha='left', va='top', fontsize=FS_ANNOT - 1)
        ax.set_xticks([-2, -1, 0, 1, 2]); ax.set_yticks([])
    for ax in axs[:2]:
        ax.set_xticklabels([])
    for ax in axs[2:]:
        ax.set_xticklabels(['', '−1', '0', '1', ''])
    fig.text(0.5 * (axs[2].get_position().x0 + axs[3].get_position().x1), axs[2].get_position().y0 - 0.17 / H,
             'E − E$_{VBM}$ (eV)', ha='center', va='top', fontsize=FS_LABEL)
    axs[0].set_ylabel('DOS', y=0.0, labelpad=2)
    # DFT handle = shaded DOS + line, as drawn
    hs = [(Patch(fc=Cd_SCF_FILL, ec='none'), Line2D([], [], color=Cd_DOS['scf']['color'], lw=Cd_DOS['scf']['lw']))]
    hs += [Line2D([], [], color=st['color'], lw=st['lw'], ls=st['ls']) for k, st in Cd_DOS.items() if k != 'scf']
    axs[1].legend(hs, [st['label'] for st in Cd_DOS.values()], loc='lower right', bbox_to_anchor=(1.0, 1.0), ncol=3, fontsize=FS_LEGEND - 1, handlelength=1.2,
                  handletextpad=0.3, columnspacing=0.6, borderaxespad=0.1)   # 放在 2 x 2 上方
    return axs[0]
panel_run(3, 'd', draw_d)


In [ ]:
# composite
fig, slots = make_layout()
for L, sp in zip('abcd', slots):
    prior = set(fig.axes)
    first = globals()['draw_' + L](fig, sp)
    new = [a for a in fig.axes if a not in prior]
    mark(fig, first if first is not None else new[0], L, dx_in=-0.3)
export_composite(fig, OUT, 'conf_fig4')
plt.close(fig)
display(NotebookImage(filename=str(OUT / 'conf_fig4.png'), width=int(W * 600 * PREVIEW_SCALE / 2)))
